In [ ]:
from google.colab import drive

# 1) Connect Colab to your Google Drive so it can see your files
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# 2) Change the current folder to where YOUR dataset lives
DATASET_DIR = "/content/drive/MyDrive/label studio"  # <-- change this if your folder name is different

os.chdir(DATASET_DIR)
print("Now we are in:", os.getcwd())

Now we are in: /content/drive/MyDrive/label studio


In [ ]:
# 3) Create the YAML file that tells YOLO where the data is

yaml_text = f"""
path: {DATASET_DIR}
train: images/train
val: images/validate
test: images/test

nc: 2
names: ['Hand', 'Gel']
"""

with open("hand_segmentation_dataset.yaml", "w") as f:
    f.write(yaml_text)

print("YAML file created at:", os.path.join(DATASET_DIR, "hand_segmentation_dataset.yaml"))
print("YAML contents:")
print(yaml_text)

YAML file created at: /content/drive/MyDrive/label studio/hand_segmentation_dataset.yaml
YAML contents:

path: /content/drive/MyDrive/label studio
train: images/train
val: images/validate
test: images/test

nc: 2
names: ['Hand', 'Gel']



In [ ]:
# 4) Install Ultralytics (YOLO) in Colab
!pip install -q ultralytics


In [ ]:
from ultralytics import YOLO
from pathlib import Path

# 5) Try creating a YOLO segmentation model (YOLOv8n-seg) and see if the dataset path is valid

yaml_path = str(Path("hand_segmentation_dataset.yaml").resolve())
print("Using dataset YAML:", yaml_path)

# Load a small segmentation model
model = YOLO("yolov8n-seg.pt")  # tiny model, good for testing

print("Model loaded OK.")

Using dataset YAML: /content/drive/MyDrive/label studio/hand_segmentation_dataset.yaml
Model loaded OK.


In [ ]:
# 6) Train a very small segmentation model on your dataset
#    We keep epochs and image size small so it runs fast.
import torch
results = model.train(
    task="segment",                 # we are doing segmentation, not just boxes
    data=yaml_path,                 # our YAML file
    epochs=5,                       # small number just to see it works
    imgsz=640,                      # image size
    batch=2,                        # how many images per step (small because you only have 26 images)
    device="0" if torch.cuda.is_available() else "cpu",  # use GPU if available
    project="hand_seg_runs",        # folder name for runs
    name="yolov8n_seg_test"         # run name
)

Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/label studio/hand_segmentation_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_test5, nbs=64, nms=False, opset=None, optimize=Fals

In [ ]:
from pathlib import Path

best_weights = Path("/content/drive/MyDrive/label studio/runs/segment/hand_seg_runs/yolov8n_seg_test4/weights/best.pt")
print(best_weights, "exists?", best_weights.exists())

/content/drive/MyDrive/label studio/runs/segment/hand_seg_runs/yolov8n_seg_test4/weights/best.pt exists? True


In [ ]:
from google.colab import files
files.download(str(best_weights))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

# 7) Find the 'best.pt' weights produced by training

run_dir = Path("hand_seg_runs") / "yolov8n_seg_test4"
best_weights = run_dir / "weights" / "best.pt"

print("Best model weights saved at:", best_weights)
print("Exists?", best_weights.exists())

Best model weights saved at: hand_seg_runs/yolov8n_seg_test4/weights/best.pt
Exists? False


In [ ]:
from google.colab import files
from pathlib import Path

best_weights = Path("/content/drive/MyDrive/label studio/runs/segment/hand_seg_runs/yolov8n_seg_test4/weights/best.pt")

print("Exists?", best_weights.exists())
files.download(str(best_weights))

Exists? True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>